# Saferoute synthetic ML engineering demo

All rows below are deterministically generated synthetic demo data. They are not real sensor readings and should not be used for safety-critical decisions. Run all cells to train, evaluate on held-out data, and save local artifacts.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
SEED = 20261118
PROVENANCE = 'synthetic_training_data'
OUT = Path('ai_models/models')
OUT.mkdir(parents=True, exist_ok=True)

## Generate labelled synthetic datasets

In [ ]:
def labelled_frame(data, labels):
    df = pd.DataFrame(data)
    df['label'] = labels
    df['provenance'] = PROVENANCE
    df['data_note'] = 'Generated synthetic engineering demo data; not real sensor data'
    return df

def gen_accident(n=2400, seed=1101):
    rng=np.random.default_rng(seed); X=rng.normal(0,1,(n,6)); y=np.full(n,'Normal movement',dtype=object)
    event=rng.random(n)<.22; X[event,:3]*=rng.uniform(3,6,(event.sum(),3)); X[event,3:]*=rng.uniform(2.5,5,(event.sum(),3)); y[event]='Possible accident'
    return labelled_frame(pd.DataFrame(X,columns=['accel_x','accel_y','accel_z','gyro_x','gyro_y','gyro_z']),y)

def gen_driving(n=2400, seed=2202):
    rng=np.random.default_rng(seed); speed=rng.uniform(0,145,n); accel=rng.normal(.1,1.8,n); brake=np.clip(rng.beta(1.4,3.5,n),0,1); harsh=rng.poisson(.45,n).astype(float)
    risky=(speed>100)|(harsh>=2)|((brake>.82)&(accel< -1.2))|(accel>4.0); y=np.where(risky,'Risky driving','Safe')
    return labelled_frame({'speed_kmh':speed,'acceleration_longitudinal':accel,'brake_intensity':brake,'harsh_brake_count':harsh},y)

def gen_route(n=2400, seed=3303):
    rng=np.random.default_rng(seed); dist=np.clip(rng.gamma(2,16,n),0,220); heading=np.clip(abs(rng.normal(10,17,n)),0,180); speed=rng.uniform(0,120,n); progress=rng.uniform(0,1,n)
    dev=(dist>45)|((heading>55)&(dist>20)); y=np.where(dev,'Route deviation','Normal route')
    return labelled_frame({'distance_from_route_m':dist,'heading_delta_deg':heading,'speed_kmh':speed,'route_progress':progress},y)

datasets={'accident_detection':(gen_accident(),['accel_x','accel_y','accel_z','gyro_x','gyro_y','gyro_z'],['Normal movement','Possible accident']), 'driver_behavior':(gen_driving(),['speed_kmh','acceleration_longitudinal','brake_intensity','harsh_brake_count'],['Safe','Risky driving']), 'route_anomaly':(gen_route(),['distance_from_route_m','heading_delta_deg','speed_kmh','route_progress'],['Normal route','Route deviation'])}
Path('datasets').mkdir(exist_ok=True)
for name,(df,features,labels) in datasets.items(): df.to_csv(Path('datasets')/(name+'.csv'),index=False)
{name: df.head() for name,(df,_,_) in datasets.items()}

## Preprocessing, stratified split, models and held-out metrics

In [ ]:
metrics={}
for name,(df,features,labels) in datasets.items():
    X_train,X_test,y_train,y_test=train_test_split(df[features],df['label'],test_size=.25,random_state=SEED,stratify=df['label'])
    model=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('classifier',RandomForestClassifier(n_estimators=160,min_samples_leaf=2,class_weight='balanced',random_state=SEED,n_jobs=1))])
    model.fit(X_train,y_train); pred=model.predict(X_test)
    p,r,f,s=precision_recall_fscore_support(y_test,pred,labels=labels,zero_division=0)
    cm=confusion_matrix(y_test,pred,labels=labels)
    metrics[name]={'accuracy':float(accuracy_score(y_test,pred)),'precision_macro':float(np.mean(p)),'recall_macro':float(np.mean(r)),'f1_macro':float(np.mean(f)),'labels':labels,'confusion_matrix':cm.tolist(),'confusion_matrix_label_order':labels,'test_rows':len(y_test),'train_rows':len(y_train)}
    display(pd.DataFrame(cm,index=labels,columns=labels).rename_axis('actual').assign(**{}))
    print(name, metrics[name])
    joblib.dump(model,OUT/(name.replace('accident_detection','accident').replace('driver_behavior','driving').replace('route_anomaly','route')+'_model.joblib'))

## Save computed evaluation metadata
Metrics are calculated above from this execution's stratified held-out split; no scores are hard-coded.

In [ ]:
metadata={'provenance':PROVENANCE,'data_note':'All rows are generated synthetic engineering demo data, not real sensor data.','random_seed':SEED,'test_size':.25,'split':'stratified train_test_split','preprocessing':'median SimpleImputer then StandardScaler in Pipeline','models':metrics}
(OUT/'notebook_metrics.json').write_text(json.dumps(metadata,indent=2)+'\n')
print('Saved models and computed metrics under',OUT)